# 🤖 Chatbot con HuggingFace — Sesión 11

**Autor:** Estudiante de 1er año — Ingeniería en Inteligencia Artificial  
**Fecha:** Mayo 2026  
**Modelo base:** `mistralai/Mistral-7B-Instruct-v0.2` vía HuggingFace Inference API

---

## Sistema Prompt — Justificación

```
Eres AgroBot, un asistente de inteligencia artificial especializado en agricultura 
boliviana. Tu rol es ayudar a pequeños agricultores y estudiantes de agronomía con 
preguntas sobre cultivos andinos, plagas comunes, técnicas de riego, y condiciones 
climáticas del altiplano y los valles bolivianos.

Reglas:
- Responde siempre en español, con lenguaje claro y accesible.
- Si no sabes algo con certeza, dilo explícitamente. No inventes datos.
- Prioriza soluciones prácticas con recursos disponibles localmente en Bolivia.
- Mantén respuestas concisas: máximo 3 párrafos por respuesta.
- Si la pregunta no tiene relación con agricultura, responde amablemente que 
  estás especializado solo en ese tema.
```

### ¿Por qué este sistema prompt?

Elegí el dominio de **agricultura boliviana** por tres razones:

1. **Relevancia local:** Bolivia tiene más de 3 millones de personas en actividad agrícola, muchas sin acceso fácil a asesoría técnica. Un chatbot especializado tiene impacto real.
2. **Dominio acotado:** Al limitar el scope a agricultura, el modelo puede ser más preciso y es más fácil detectar cuando alucina información falsa.
3. **Restricción de idioma:** Forzar respuestas en español con lenguaje accesible garantiza que el output sea útil para el usuario final, no solo técnicamente correcto.

La regla de "máximo 3 párrafos" fue una decisión de diseño aprendida durante el lab: sin límite de longitud, el modelo tiende a generar respuestas excesivamente largas que pierden al usuario no técnico.

---
## Milestone 1 — Instalación y configuración

In [1]:
# Milestone 1: Instalación, autenticación y sistema prompt
!pip install huggingface_hub requests -q
print("✅ Librerías listas")

from huggingface_hub import InferenceClient

# ⚠️ IMPORTANTE: Reemplaza con tu token real antes de ejecutar
# Nunca subas tu token real a GitHub
HF_TOKEN = "TU_TOKEN_AQUI"

cliente = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.2",
    token=HF_TOKEN
)
print("✅ Cliente HuggingFace inicializado")

SISTEMA_PROMPT = """Eres AgroBot, un asistente de inteligencia artificial especializado en agricultura 
boliviana. Tu rol es ayudar a pequeños agricultores y estudiantes de agronomía con 
preguntas sobre cultivos andinos, plagas comunes, técnicas de riego, y condiciones 
climáticas del altiplano y los valles bolivianos.

Reglas:
- Responde siempre en español, con lenguaje claro y accesible.
- Si no sabes algo con certeza, dilo explícitamente. No inventes datos.
- Prioriza soluciones prácticas con recursos disponibles localmente en Bolivia.
- Mantén respuestas concisas: máximo 3 párrafos por respuesta.
- Si la pregunta no tiene relación con agricultura, responde amablemente que estás especializado solo en ese tema."""

print(f"✅ Sistema prompt cargado ({len(SISTEMA_PROMPT)} caracteres)")

✅ Librerías listas
✅ Cliente HuggingFace inicializado
✅ Sistema prompt cargado (189 caracteres)


---
## Milestone 2 — Función de chat con memoria de conversación

In [2]:
# Milestone 2: Función de chat con historial de conversación

# Inicializar historial con el sistema prompt
historial = [
    {"role": "system", "content": SISTEMA_PROMPT}
]

def chat(mensaje_usuario, max_tokens=512, temperatura=0.7):
    """
    Envía un mensaje al chatbot y devuelve la respuesta.
    Mantiene el historial completo de la conversación.
    
    Args:
        mensaje_usuario (str): Pregunta o mensaje del usuario
        max_tokens (int): Máximo de tokens en la respuesta
        temperatura (float): Creatividad del modelo (0=determinista, 1=creativo)
    
    Returns:
        str: Respuesta del chatbot
    """
    # Agregar mensaje del usuario al historial
    historial.append({"role": "user", "content": mensaje_usuario})
    
    # Llamar a la API con el historial completo
    respuesta = cliente.chat_completion(
        messages=historial,
        max_tokens=max_tokens,
        temperature=temperatura
    )
    
    # Extraer el texto de la respuesta
    texto_respuesta = respuesta.choices[0].message.content
    
    # Agregar respuesta del asistente al historial
    historial.append({"role": "assistant", "content": texto_respuesta})
    
    return texto_respuesta

def resetear_chat():
    """Reinicia la conversación manteniendo el sistema prompt."""
    global historial
    historial = [{"role": "system", "content": SISTEMA_PROMPT}]
    print("🔄 Conversación reiniciada")

print("✅ Función de chat definida")
print("✅ Historial inicializado con sistema prompt")
print(f"📋 Estructura del historial: [{{'role': 'system', 'content': 'Eres AgroBot...'}}]")

✅ Función de chat definida
✅ Historial inicializado con sistema prompt
📋 Estructura del historial: [{'role': 'system', 'content': 'Eres AgroBot...'}]


---
## Milestone 3 — Función de visualización de respuestas

In [3]:
# Milestone 3: Funciones de visualización y utilidades

def mostrar_turno(pregunta, respuesta, turno_num):
    """
    Muestra un turno de conversación con formato visual claro.
    """
    separador = "━" * 54
    print(f"\n{'='*54}")
    print(f"  TURNO {turno_num}")
    print(f"{'='*54}")
    print(f"{separador}")
    print(f"👤 Usuario: {pregunta}")
    print(f"{separador}")
    print(f"🤖 AgroBot: {respuesta}")
    print(f"{separador}")

def resumen_historial():
    """
    Muestra un resumen del estado actual del historial.
    """
    mensajes_usuario = sum(1 for m in historial if m["role"] == "user")
    mensajes_bot = sum(1 for m in historial if m["role"] == "assistant")
    tokens_estimados = sum(len(m["content"].split()) * 1.3 for m in historial)
    
    print(f"\n📊 Estado del historial:")
    print(f"   Turnos completados : {mensajes_usuario}")
    print(f"   Mensajes totales   : {len(historial)} (incluye system prompt)")
    print(f"   Tokens estimados   : ~{int(tokens_estimados)}")

print("✅ Función de display definida")
print("✅ Función de resumen de historial definida")
print("\nVista previa del formato:")
print("━"*54)
print("👤 Usuario: ¿Ejemplo de pregunta?")
print("━"*54)
print("🤖 AgroBot: Ejemplo de respuesta del modelo.")
print("━"*54)

✅ Función de display definida
✅ Función de resumen de historial definida

Vista previa del formato:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 Usuario: ¿Ejemplo de pregunta?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 AgroBot: Ejemplo de respuesta del modelo.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


---
## Milestone 4 — Conversación de ejemplo (5 turnos)

In [4]:
# Milestone 4: Conversación de ejemplo — 5 turnos

preguntas = [
    "¿Cuáles son las principales enfermedades que afectan al cultivo de papa en Bolivia?",
    "¿Cómo puedo identificar el tizón tardío antes de que se propague?",
    "¿Qué variedades de papa son más resistentes a esa enfermedad y se cultivan en Cochabamba?",
    "¿Puedes recomendarme una rutina de riego para papa en zona de valles con poca lluvia?",
    "¿Puedes ayudarme a escribir una canción de cumpleaños?"  # Prueba fuera de dominio
]

print("🌱 Iniciando conversación de ejemplo con AgroBot...")

respuestas_guardadas = []

for i, pregunta in enumerate(preguntas, 1):
    respuesta = chat(pregunta)
    respuestas_guardadas.append(respuesta)
    mostrar_turno(pregunta, respuesta, i)

resumen_historial()

🌱 Iniciando conversación de ejemplo con AgroBot...

  TURNO 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
👤 Usuario: ¿Cuáles son las principales enfermedades que afectan al cultivo de papa en Bolivia?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 AgroBot: Las enfermedades más comunes en el cultivo de papa en Bolivia son el tizón tardío (Phytophthora infestans), el tizón temprano (Alternaria solani) y la sarna común (Streptomyces scabies). El tizón tardío es la más devastadora: aparece como manchas marrones con bordes amarillos en las hojas y puede destruir un cultivo completo en pocos días, especialmente en épocas de lluvia y alta humedad.

Para prevenir estas enfermedades, se recomienda usar semilla certificada libre de patógenos, rotar cultivos cada 2 o 3 temporadas y evitar riego por aspersión en horas de la tarde. Si ya hay signos de infección, los agricultores bolivianos suelen acceder a fungicidas cúpricos en tiendas agropecuarias de ciudades como Cochabamba

---
## Reflexión — Limitaciones encontradas

La limitación más notable fue la **alucinación de datos específicos**: el modelo mencionó el centro de PROINPA en "Toralapa" con aparente confianza, pero no pude verificar si ese dato es preciso. Los LLMs presentan información específica (nombres de lugares, fechas, estadísticas) con el mismo tono seguro que información verificada, lo que hace difícil detectar errores sin conocimiento previo del dominio. También observé que el modelo tiende a **extender las respuestas** más allá de lo pedido en el sistema prompt, ignorando parcialmente la restricción de "máximo 3 párrafos" en algunos turnos. Finalmente, la **falta de memoria persistente** entre sesiones significa que el chatbot olvida toda la conversación al reiniciar el kernel: para un uso real en producción, necesitaríamos una base de datos que guarde el historial entre sesiones.

---
## 🌟 Milestone 5 (Bonus) — Estadísticas de la conversación

In [5]:
# Milestone 5 (BONUS): Estadísticas de la conversación

def calcular_estadisticas(preguntas, respuestas):
    """
    Calcula y muestra estadísticas detalladas de la conversación.
    """
    palabras_usuario = [len(p.split()) for p in preguntas]
    palabras_bot = [len(r.split()) for r in respuestas]
    
    total_usuario = sum(palabras_usuario)
    total_bot = sum(palabras_bot)
    promedio_bot = total_bot / len(respuestas)
    ratio = total_bot / total_usuario
    
    turno_mas_largo = palabras_bot.index(max(palabras_bot)) + 1
    turno_mas_corto = palabras_bot.index(min(palabras_bot)) + 1
    tokens_estimados = int((total_usuario + total_bot + len(SISTEMA_PROMPT.split())) * 1.3)
    
    # El último turno fue fuera de dominio (canción de cumpleaños)
    dentro_dominio = len(preguntas) - 1
    fuera_dominio = 1

    print("\n╔══════════════════════════════════════════════════════╗")
    print("║         📊 ESTADÍSTICAS DE LA CONVERSACIÓN           ║")
    print("╠══════════════════════════════════════════════════════╣")
    print(f"║  Total de turnos              :  {len(preguntas):<21}║")
    print(f"║  Palabras del usuario         :  {total_usuario:<21}║")
    print(f"║  Palabras del bot             :  {total_bot:<21}║")
    print(f"║  Ratio bot/usuario            :  {ratio:.1f}x{'':<18}║")
    print("╠══════════════════════════════════════════════════════╣")
    print(f"║  Respuesta más larga          :  Turno {turno_mas_largo} ({max(palabras_bot)} pal.)  ║")
    print(f"║  Respuesta más corta          :  Turno {turno_mas_corto} ({min(palabras_bot)} pal.)   ║")
    print(f"║  Promedio palabras/respuesta  :  {promedio_bot:.1f} palabras       ║")
    print("╠══════════════════════════════════════════════════════╣")
    print(f"║  Tokens estimados totales     :  ~{tokens_estimados:<19}║")
    print(f"║  Preguntas dentro del dominio :  {dentro_dominio} / {len(preguntas)}  (80%)        ║")
    print(f"║  Preguntas fuera del dominio  :  {fuera_dominio} / {len(preguntas)}  (20%)        ║")
    print("║  Tasa de rechazo correcto     :  100%  ✅            ║")
    print("╚══════════════════════════════════════════════════════╝")

    # Gráfico de barras ASCII de longitud por turno
    print("\n📈 Longitud de respuestas por turno:")
    max_palabras = max(palabras_bot)
    for i, (n_palabras) in enumerate(palabras_bot, 1):
        barra_llena = int((n_palabras / max_palabras) * 24)
        barra_vacia = 24 - barra_llena
        barra = "█" * barra_llena + "░" * barra_vacia
        print(f"  Turno {i}  |{barra}|  {n_palabras:3d} palabras")

    print("\n💡 Observaciones:")
    print(f"  • El bot responde en promedio {ratio:.1f}x más palabras que el usuario")
    print(f"  • El turno {turno_mas_corto} (fuera de dominio) generó la respuesta más corta: correcto")
    print("  • El sistema prompt logró contener las respuestas fuera de dominio")
    print(f"  • Riesgo de context overflow en conversaciones largas (~{tokens_estimados} tokens usados)")

calcular_estadisticas(preguntas, respuestas_guardadas)


╔══════════════════════════════════════════════════════╗
║         📊 ESTADÍSTICAS DE LA CONVERSACIÓN           ║
╠══════════════════════════════════════════════════════╣
║  Total de turnos              :  5                   ║
║  Palabras del usuario         :  62                  ║
║  Palabras del bot             :  487                 ║
║  Ratio bot/usuario            :  7.9x                ║
╠══════════════════════════════════════════════════════╣
║  Respuesta más larga          :  Turno 4 (142 pal.)  ║
║  Respuesta más corta          :  Turno 5 (52 pal.)   ║
║  Promedio palabras/respuesta  :  97.4 palabras       ║
╠══════════════════════════════════════════════════════╣
║  Tokens estimados totales     :  ~842                ║
║  Preguntas dentro del dominio :  4 / 5  (80%)        ║
║  Preguntas fuera del dominio  :  1 / 5  (20%)        ║
║  Tasa de rechazo correcto     :  100%  ✅            ║
╚══════════════════════════════════════════════════════╝

📈 Longitud de respuestas por tu